In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

base_path = "/content/drive/MyDrive/files"

folders = [f for f in os.listdir(base_path) if f.startswith("S")]

filenumber = ['03','07','11']
data = []

for folder in sorted(folders):
    folder_path = os.path.join(base_path, folder)

    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)

    for f in files:
        if f.lower().endswith(".edf"):

            run_part = f.split("R")[-1].split(".")[0]

            if run_part in filenumber:
                full_path = os.path.join(folder_path, f)
                data.append(full_path)

# Print final single list
print(data)
print("\nTotal files:", len(data))


['/content/drive/MyDrive/files/S071/S071R03.edf', '/content/drive/MyDrive/files/S071/S071R11.edf', '/content/drive/MyDrive/files/S071/S071R07.edf', '/content/drive/MyDrive/files/S072/S072R11.edf', '/content/drive/MyDrive/files/S072/S072R07.edf', '/content/drive/MyDrive/files/S072/S072R03.edf', '/content/drive/MyDrive/files/S074/S074R07.edf', '/content/drive/MyDrive/files/S074/S074R03.edf', '/content/drive/MyDrive/files/S074/S074R11.edf', '/content/drive/MyDrive/files/S076/S076R03.edf', '/content/drive/MyDrive/files/S076/S076R11.edf', '/content/drive/MyDrive/files/S076/S076R07.edf', '/content/drive/MyDrive/files/S077/S077R03.edf', '/content/drive/MyDrive/files/S077/S077R07.edf', '/content/drive/MyDrive/files/S077/S077R11.edf', '/content/drive/MyDrive/files/S078/S078R11.edf', '/content/drive/MyDrive/files/S078/S078R07.edf', '/content/drive/MyDrive/files/S078/S078R03.edf', '/content/drive/MyDrive/files/S079/S079R07.edf', '/content/drive/MyDrive/files/S079/S079R03.edf', '/content/drive/MyD

In [3]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 51.8 MB/s eta 0:00:00


In [4]:
import mne

raw_list = []

for path in data:
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw_list.append(raw)

print("Total raw files loaded:", len(raw_list))

/tmp/ipython-input-174/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipython-input-174/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipython-input-174/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


Total raw files loaded: 111


In [5]:
import mne

data = {}

for i, raw in enumerate(raw_list):

    subject_key = f"subject_{i+1}"
    raw.set_eeg_reference('average')
    data[subject_key] = {}
    events, event_dict = mne.events_from_annotations(raw, verbose=False)
    epochs = mne.Epochs(
        raw,
        events,
        event_id={'T0': 1, 'T1': 2, 'T2': 3},
        tmin=0.5,
        tmax=2.5,
        baseline=None,
        preload=True,
        verbose=False
    )
    epochs.pick_types(eeg=True)
    data[subject_key]['T0'] = epochs['T0'].get_data()
    data[subject_key]['T1'] = epochs['T1'].get_data()
    data[subject_key]['T2'] = epochs['T2'].get_data()

    print(f"{subject_key} extracted:")
    print("T0 shape:", data[subject_key]['T0'].shape)
    print("T1 shape:", data[subject_key]['T1'].shape)
    print("T2 shape:", data[subject_key]['T2'].shape)
    print("-------------")

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
subject_1 extracted:
T0 shape: (15, 64, 321)
T1 shape: (7, 64, 321)
T2 shape: (8, 64, 321)
-------------
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
subject_2 extracted:
T0 shape: (15, 64, 321)
T1 shape: (8, 64, 321)
T2 shape: (7, 64, 321)
-------------
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
subject_3 extracted:
T0 shape: (15, 64, 321)
T1 shape: (8, 64, 321)
T2 shape: (7, 64, 321)
-------------
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick

In [9]:
import numpy as np

subject = "subject_10"

X_t1 = data[subject]['T1']   # shape (trials, channels, time)
X_t2 = data[subject]['T2']

X = np.concatenate([X_t1, X_t2], axis=0)

y = np.array([0]*len(X_t1) + [1]*len(X_t2))

print("Data shape:", X.shape)
print("Labels shape:", y.shape)

Data shape: (15, 64, 321)
Labels shape: (15,)


In [10]:
from mne.filter import filter_data

sfreq = raw_list[0].info['sfreq']

X = filter_data(X, sfreq, 8., 30., verbose=False)

In [11]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

csp = CSP(n_components=4, reg='ledoit_wolf', log=True)

lda = LinearDiscriminantAnalysis()

clf = Pipeline([
    ('CSP', csp),
    ('LDA', lda)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(clf, X, y, cv=cv)

print("Subject 1 Accuracy: %.2f%%" % (np.mean(scores)*100))

Computing rank from data with rank=None
    Using tolerance 2.1e-05 (2.2e-16 eps * 64 dim * 1.4e+09  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolerance 2e-05 (2.2e-16 eps * 64 dim * 1.4e+09  max singular value)
    Estimated rank (data): 63
    data: rank 63 computed from 64 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
    Setting small data eigenvalues to zero (without PCA)
Computing rank from data with rank=None
    Using tolera

In [13]:
import numpy as np
import mne

erd_data = {}

for i, raw in enumerate(raw_list):

    subject_key = f"subject_{i+1}"
    erd_data[subject_key] = {}

    raw.set_eeg_reference('average', verbose=False)
    raw.filter(8., 30., verbose=False)

    events, _ = mne.events_from_annotations(raw, verbose=False)

    epochs = mne.Epochs(
        raw,
        events,
        event_id={'T0':1, 'T1':2, 'T2':3},
        tmin=0.5,
        tmax=2.5,
        baseline=None,
        preload=True,
        verbose=False
    )

    epochs.pick_types(eeg=True)

    # ----------------------
    # Baseline (T0)
    # ----------------------
    T0 = epochs['T0'].get_data()
    baseline_var = np.mean(np.var(T0, axis=2), axis=0)
    # shape: (channels,)

    # ----------------------
    # T1
    # ----------------------
    T1 = epochs['T1'].get_data()
    var_T1 = np.var(T1, axis=2)
    ERD_T1 = (var_T1 - baseline_var) / baseline_var

    # ----------------------
    # T2
    # ----------------------
    T2 = epochs['T2'].get_data()
    var_T2 = np.var(T2, axis=2)
    ERD_T2 = (var_T2 - baseline_var) / baseline_var

    erd_data[subject_key]['T1'] = ERD_T1
    erd_data[subject_key]['T2'] = ERD_T2

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).

In [15]:
import numpy as np

X_all = []
y_all = []

for subject in erd_data.keys():

    X_t1 = erd_data[subject]['T1']
    X_t2 = erd_data[subject]['T2']

    X_all.append(np.vstack([X_t1, X_t2]))
    y_all.append(np.array([0]*len(X_t1) + [1]*len(X_t2)))

X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)

print("Shape:", X_all.shape)

Shape: (1687, 64)


In [16]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold

lda = LinearDiscriminantAnalysis()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lda, X_all, y_all, cv=cv)

print("ERD + LDA Accuracy: %.2f%%" % (scores.mean()*100))

ERD + LDA Accuracy: 60.23%


In [17]:
!pip install pyriemann

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 6.5 MB/s eta 0:00:00


In [19]:
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

model = Pipeline([
    ('Cov', Covariances(estimator='oas')),  # <-- IMPORTANT
    ('TS', TangentSpace()),
    ('LDA', LinearDiscriminantAnalysis())
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv)

print("Riemannian Accuracy: %.2f%%" % (scores.mean()*100))

Riemannian Accuracy: 46.67%


In [28]:
import numpy as np

sfreq = 160
duration = 4
n_samples = sfreq * duration
n_channels = 64

t = np.arange(n_samples) / sfreq

# Base EEG noise
signal = np.random.randn(n_channels, n_samples) * 0.05

# 10 Hz mu rhythm
mu_wave = np.sin(2 * np.pi * 10 * t)

# Find real channel indices from your data
# print(epochs.ch_names)
# c3_index = epochs.ch_names.index('C3..')
# c4_index = epochs.ch_names.index('C4..')

c3_index = 8    # replace with correct index
c4_index = 10   # replace with correct index

# Simulate LEFT imagery:
# stronger suppression in C4
signal[c4_index] += mu_wave * 0.2
signal[c3_index] += mu_wave * 0.6

print("4-sec signal shape:", signal.shape)

4-sec signal shape: (64, 640)


In [25]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold

lda = LinearDiscriminantAnalysis()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lda, X_all, y_all, cv=cv)

print("ERD + LDA Accuracy: %.2f%%" % (scores.mean()*100))

ERD + LDA Accuracy: 60.23%


In [26]:
lda = LinearDiscriminantAnalysis()

# Train on full dataset
lda.fit(X_all, y_all)

LinearDiscriminantAnalysis()

In [31]:
import numpy as np

n_channels = 64

# baseline random small ERD noise
erd_left = np.random.normal(0, 0.05, n_channels)

# simulate stronger ERD at C4
c3_index = 8
c4_index = 10

erd_left[c4_index] = -0.35   # strong ERD
erd_left[c3_index] = -0.15   # weaker ERD

# reshape to (1, channels)
erd_left = erd_left.reshape(1, -1)

print("LEFT ERD shape:", erd_left.shape)
erd_right = np.random.normal(0, 0.05, n_channels)

erd_right[c3_index] = -0.35
erd_right[c4_index] = -0.15

erd_right = erd_right.reshape(1, -1)

print("RIGHT ERD shape:", erd_right.shape)

LEFT ERD shape: (1, 64)
RIGHT ERD shape: (1, 64)


In [32]:
pred_left = lda.predict(erd_left)
pred_right = lda.predict(erd_right)

print("Prediction LEFT:", pred_left)
print("Prediction RIGHT:", pred_right)

Prediction LEFT: [0]
Prediction RIGHT: [1]


In [34]:
from google.colab import drive

joblib.dump(lda, "/content/drive/MyDrive/lda_erd_model.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['/content/drive/MyDrive/lda_erd_model.pkl']